# Exploration

Using the font fix, then printing the amount of sheets

In [2]:
from openpyxl.styles.fonts import Font
Font.family.max = 100          # MUST run before load_workbook

from openpyxl import load_workbook

wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                   read_only=True, data_only=True)
print(len(wb.sheetnames))      # expect 94

94


### Discover the sheet names

In [3]:
for i, name in enumerate(wb.sheetnames):
    print(i, repr(name))

0 'Weight J'
1 'Weight J (2)'
2 'Weight OR'
3 'Costs'
4 'Blank Simplified'
5 'Blank Week'
6 '14Oct24'
7 '4Nov24'
8 'Sheet1 (2)'
9 '19Nov24'
10 '25Nov24'
11 '2Dec24'
12 '9Dec24'
13 '16Dec24'
14 '30Dec24'
15 '6Jan24'
16 '20Jan24'
17 '27Jan25'
18 '03Feb25'
19 '10Feb25'
20 '17Feb25'
21 '24Feb25'
22 '03Mar25'
23 '10Mar25'
24 '17Mar25'
25 '24Mar25'
26 '31Mar25'
27 '07Apr25'
28 '14Apr25'
29 '21Apr25'
30 '28Apr25'
31 '05May25'
32 '13May25'
33 '19May25'
34 '26May25'
35 '02June25'
36 '09June25'
37 '30JUN25'
38 '07JUL25'
39 '14JUL25'
40 'Blank Schedule (2)'
41 'Blank (2)'
42 '11AUG25'
43 '18AUG25'
44 '25AUG25'
45 '01SEP25'
46 '08SEP25'
47 '15SEP25'
48 '06OCT25'
49 '13OCT25'
50 '20OCT25'
51 '27OCT25'
52 '03NOV25'
53 'IGNORE'
54 '10NOV25'
55 '17NOV25'
56 '24NOV25'
57 '01DEC25'
58 '08DEC25'
59 '29DEC25'
60 '05JAN26'
61 '12JAN26'
62 '19JAN26'
63 '26Jan26'
64 '02Feb26'
65 '09Feb26'
66 '16Feb26'
67 '23Feb26'
68 '02Mar26'
69 '09Mar26'
70 '16Mar26'
71 '23Mar26'
72 '30Mar26'
73 '07APR26'
74 '13APR26'
75 '

### Find a given input in a row

In [4]:
def find_header_rows(sheet_name, target="Time"): #defining our function so we can reuse, sets both parameters of sheet_name and target="Time" as default, though we can use this to find any value.
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                   read_only=True, data_only=True) #we have to reload the workbook here because we can only read each sheet once

    ws = wb[sheet_name] #selecting our sheet 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True),start=1): #loop which runs row by row, enumerating each cell with row number and cell value
        time_columns = [] #starting an empty list
        for column_number, value in enumerate(cell_values): #second loop which enumerates the column number and value for each given cell
            if str(value).strip() == target: #takes the stripped output of the string and if it is equal to target parameter...
                time_columns.append(column_number) #...then the corresponding column_number is appended to the time_columns set
        if time_columns: #if there is something in time_columns...
            print("row", row_number, "->", time_columns) #...then print the given text and row_number and columns that time is in


find_header_rows("06JUL26", "Total") #passing the function our two arguments here
find_header_rows("06JUL26", "Time")

find_header_rows("20APR26", "Total")
find_header_rows("20APR26", "Time")

find_header_rows("02Feb26", "Total")
find_header_rows("02Feb26", "Time")


row 26 -> [2, 11]
row 54 -> [2, 11]
row 81 -> [2, 11]
row 108 -> [2]
row 3 -> [2, 11]
row 31 -> [2, 11]
row 58 -> [2, 11]
row 85 -> [2]
row 26 -> [2, 11]
row 54 -> [2, 11]
row 81 -> [2, 11]
row 108 -> [2]
row 3 -> [2, 11]
row 31 -> [2, 11]
row 58 -> [2, 11]
row 85 -> [2]
row 26 -> [2, 11]
row 54 -> [2, 11]
row 81 -> [2, 11]
row 110 -> [2]
row 3 -> [2, 11]
row 31 -> [2, 11]
row 58 -> [2, 11]
row 87 -> [2]


This confirms the locations for both "Time" and "Total", which tells me the layout of the sheet. In row 26, "Total" is in column 2 and 11, and so on. In row 3, "Total" is in column 2 and 11, and so on.

02Feb26 band 4 starts at row 87, not 85. Total at 110, not 108.

Confirms that Era 3 Bands (each multiple of rows containing a given set of days, like Monday+Tuesday, Wednesday+Thursday, etc) are standard on 1-3, but differ for Band 4 (Sunday)

Necessitates the parser to account for this difference, cannot hardcode for 4 header rows as 3, 31, 58, and 85.

Running the function on some era 1 and era 2 sheets.

In [5]:
find_header_rows("2Dec24", "Time")
find_header_rows("2Dec24", "Total")

find_header_rows("13OCT25", "Time")
find_header_rows("13OCT25", "Total")

row 2 -> [1, 6]
row 21 -> [1, 6]
row 40 -> [1, 6]
row 59 -> [1]
row 18 -> [1, 6]
row 37 -> [1, 6]
row 56 -> [1, 6]
row 75 -> [1]
row 3 -> [2, 10]
row 31 -> [2, 10]
row 58 -> [2, 10]
row 87 -> [2]
row 12 -> [21]
row 26 -> [2, 10]
row 54 -> [2, 10]
row 81 -> [2, 10]
row 110 -> [2]


This proves that the parser must be dynamic for multiple of our values. Header rows, columns, block height and band spacing cannot be assumed.

Era 2 carries a stray "Total" at row 12 column 21, outside the block columns. Pair headers to totals by column, not by proximity, and assert four of each per sheet.

### Print the values of the inputted row

In [6]:

def print_header_values(sheet_name,target_row):
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                    read_only=True, data_only=True) #we have to reload the workbook here because we can only read each sheet once

    ws = wb[sheet_name] 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True),start=1): 
        if row_number == target_row:
            for column_number, value in enumerate(cell_values): 
                print("Row:", row_number,"| Column:", column_number,"| Values:", value)
            break


print_header_values("06JUL26", 3)



    

Row: 3 | Column: 0 | Values: None
Row: 3 | Column: 1 | Values: None
Row: 3 | Column: 2 | Values: Time
Row: 3 | Column: 3 | Values: Food
Row: 3 | Column: 4 | Values: Quantity (g or count)
Row: 3 | Column: 5 | Values: Calories
Row: 3 | Column: 6 | Values: Protein
Row: 3 | Column: 7 | Values: Fiber
Row: 3 | Column: 8 | Values: Carbs
Row: 3 | Column: 9 | Values: Fat
Row: 3 | Column: 10 | Values: None
Row: 3 | Column: 11 | Values: Time
Row: 3 | Column: 12 | Values: Food
Row: 3 | Column: 13 | Values: Quantity (g or count)
Row: 3 | Column: 14 | Values: Calories
Row: 3 | Column: 15 | Values: Protein
Row: 3 | Column: 16 | Values: Fiber
Row: 3 | Column: 17 | Values: Carbs
Row: 3 | Column: 18 | Values: Fat
Row: 3 | Column: 19 | Values: None
Row: 3 | Column: 20 | Values: None
Row: 3 | Column: 21 | Values: None
Row: 3 | Column: 22 | Values: None
Row: 3 | Column: 23 | Values: None
Row: 3 | Column: 24 | Values: None
Row: 3 | Column: 25 | Values: None
Row: 3 | Column: 26 | Values: None
Row: 3 | Column

In [7]:
print_header_values("13OCT25", 3)

Row: 3 | Column: 0 | Values: None
Row: 3 | Column: 1 | Values: None
Row: 3 | Column: 2 | Values: Time
Row: 3 | Column: 3 | Values: Food
Row: 3 | Column: 4 | Values: Calories
Row: 3 | Column: 5 | Values: Protein
Row: 3 | Column: 6 | Values: Fiber
Row: 3 | Column: 7 | Values: Carbs
Row: 3 | Column: 8 | Values: Fat
Row: 3 | Column: 9 | Values: None
Row: 3 | Column: 10 | Values: Time
Row: 3 | Column: 11 | Values: Food
Row: 3 | Column: 12 | Values: Calories
Row: 3 | Column: 13 | Values: Protein
Row: 3 | Column: 14 | Values: Fiber
Row: 3 | Column: 15 | Values: Carbs
Row: 3 | Column: 16 | Values: Fat
Row: 3 | Column: 17 | Values: None
Row: 3 | Column: 18 | Values: None
Row: 3 | Column: 19 | Values: None
Row: 3 | Column: 20 | Values: None
Row: 3 | Column: 21 | Values: None
Row: 3 | Column: 22 | Values: None
Row: 3 | Column: 23 | Values: None
Row: 3 | Column: 24 | Values: None


Era 2

In [8]:
print_header_values("2Dec24", 2)

Row: 2 | Column: 0 | Values: None
Row: 2 | Column: 1 | Values: Time
Row: 2 | Column: 2 | Values: Food
Row: 2 | Column: 3 | Values: Calories
Row: 2 | Column: 4 | Values: Protein
Row: 2 | Column: 5 | Values: None
Row: 2 | Column: 6 | Values: Time
Row: 2 | Column: 7 | Values: Food
Row: 2 | Column: 8 | Values: Calories
Row: 2 | Column: 9 | Values: Protein


Era 1

Differences between the three sheets are now clear, with the labels differing. Still assuming there are 3 Eras, however it is possible there are more than that. Will have to walk each sheet and test for this to properly make a determination.

In [9]:
def altered_find_header_rows(sheet_name, target="Time"):
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                   read_only=True, data_only=True) 

    ws = wb[sheet_name] 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True, max_row=15),start=1): #loop walks each row and enumerates it
        stripped_cell_values=[]

        for row_value in cell_values:
            stripped_cell_values.append(str(row_value).strip())
            
        if target in stripped_cell_values: 
            return row_number, cell_values

#This function should return the number of and values and on the row which contains the target. If the row contains no target, it does not return it and continues, if the loop ends without a match, the function returns None.


In [10]:
print(altered_find_header_rows("06JUL26"))
print(altered_find_header_rows("2Dec24"))
print(altered_find_header_rows("Weight J"))

(3, (None, None, 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', None, 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None))
(2, (None, 'Time', 'Food', 'Calories', 'Protein', None, 'Time', 'Food', 'Calories', 'Protein'))
None


In [11]:
def altered_print_header_values(sheet_name,target_row):
    wb = load_workbook("../data/raw/Nutrition141025NEW.xlsx",
                    read_only=True, data_only=True) #we have to reload the workbook here because we can only read each sheet once

    ws = wb[sheet_name] 

    for row_number, cell_values in enumerate(ws.iter_rows(values_only=True),start=1): 
        if row_number == target_row:
            for column_number, value in enumerate(cell_values): 
                print("Row:", row_number,"| Column:", column_number,"| Values:", value)
            break



In [12]:
not_a_week=[]
is_a_week=[]
#created our two lists, for weeks and non weeks.

for name in wb.sheetnames: #for loop runs, with var name being a given workbook's sheetname
    result=altered_find_header_rows(name) #var result is equal to the result the function running on each sheetname
    if result is None: #if the result is None
        not_a_week.append(name) #the name of the sheet is added to not_a_week
    else:
        is_a_week.append(name)#otherwise it is added to is_a_week

print(len(is_a_week), "logs, and ", len(not_a_week), "not logs")  #prints the text alongside specified and uses len to count the amount of entries in each list.          

print(not_a_week)
print(is_a_week)


90 logs, and  4 not logs
['Weight J', 'Weight J (2)', 'Weight OR', 'Costs']
['Blank Simplified', 'Blank Week', '14Oct24', '4Nov24', 'Sheet1 (2)', '19Nov24', '25Nov24', '2Dec24', '9Dec24', '16Dec24', '30Dec24', '6Jan24', '20Jan24', '27Jan25', '03Feb25', '10Feb25', '17Feb25', '24Feb25', '03Mar25', '10Mar25', '17Mar25', '24Mar25', '31Mar25', '07Apr25', '14Apr25', '21Apr25', '28Apr25', '05May25', '13May25', '19May25', '26May25', '02June25', '09June25', '30JUN25', '07JUL25', '14JUL25', 'Blank Schedule (2)', 'Blank (2)', '11AUG25', '18AUG25', '25AUG25', '01SEP25', '08SEP25', '15SEP25', '06OCT25', '13OCT25', '20OCT25', '27OCT25', '03NOV25', 'IGNORE', '10NOV25', '17NOV25', '24NOV25', '01DEC25', '08DEC25', '29DEC25', '05JAN26', '12JAN26', '19JAN26', '26Jan26', '02Feb26', '09Feb26', '16Feb26', '23Feb26', '02Mar26', '09Mar26', '16Mar26', '23Mar26', '30Mar26', '07APR26', '13APR26', '20APR26', '27APR26', '04MAY26', '11MAY26', '18MAY26', '25MAY26', 'College Sample', 'Blank Sheet Official', '01JUN26'

This number contains all valid logs, as well as blank templates, unused weekly sheets, and projections.
Cannot rely on this as a final test of valid sheets, only as a test of which sheets are and are not weekly logs of some form.
Blanks and templates are of lowest concern, can be detected. However, projections cannot, and contain synthetic data which could corrupt future results.

Non-real weeks: Blank Simplified, Blank Week, Sheet1 (2), Blank Schedule (2), Blank (2), IGNORE, College Sample, Blank Sheet Official, IGNORE(2)

In [13]:
r"(\d{1,2})([A-Za-z]+)(\d{2})$" #regular expression we will use to describe the shape of our valid sheet names
#\d is any single digit, 0-9
#{1,2} is one or two of the thing before it, in our case any single digit 0-9
#[A-Za-z] is any single letter, uppercase or lowercase
#+ is one or more of the thing before it (our single letter)
#{2}, exactly 2 of the thing before it (single digit 0-9)
#$ end of the string
#parentheses means to capture this part, allowing us to use it later.
#r makes this into a raw string, which is necessary as regex uses backslashes a lot, which would cause us to run into issues.

#In relation to both Excel and SQL, similar in concept, different in syntax. My familiarity at this stage is low.

'(\\d{1,2})([A-Za-z]+)(\\d{2})$'

### Test the Names of Sheets against a regex. Find the ones which do not match the given pattern (dates)

In [14]:
import re #importing regular expressions

pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" 

refined_weeks=[]

for name in is_a_week:
    nameMatching = re.match(pattern, name.strip())
    if nameMatching is None:
        print(repr(name))
    else: 
        refined_weeks.append(name)

print(len(refined_weeks))

#loop over is_a_week
#similar to is None used earlier, test should be for if the name matches the pattern
#If it does not match, it should be put into a different list
#the pattern needs to be given to the loop so it goes over the whole thing

'Blank Simplified'
'Blank Week'
'Sheet1 (2)'
'Blank Schedule (2)'
'Blank (2)'
'IGNORE'
'College Sample'
'Blank Sheet Official'
'IGNORE(2)'
81


Importing regular expressions, then using it with the given pattern, then using a similar loop from earlier, we walk through each sheet and see if it matches the pattern. If it doesn't, the stripped representation is printed so whitespace doesn't cause a failure. 

If it gets to else, it is put into refined_weeks, our confirmed list of genuine weeks.

In [15]:
from datetime import datetime, timedelta

# %b will allow us to get any abbreviated Month names 
# %B will get any full ones
# %y should parse the 2 digit years.
# %d is day of the month, zero padded like 01, 02
# %-d is day of the month with no zero padding like 1, 2

d_raw="02June25"
pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" 
m = re.match(pattern, d_raw)

In [16]:
date_raw="06JUL26"
pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" 
m = re.match(pattern, date_raw)

day_test=m.group(1)
month_test=m.group(2)
year_test=m.group(3)

month_normal=month_test[:3]

normalised_date=day_test+month_normal+year_test



### Dictionaries for: override dates, labels
### List for canon name ordering

In [17]:

OVERRIDES= {
"6Jan24": datetime(2025, 1, 6),     
"20Jan24": datetime(2025, 1, 20),  
}
#both sit between 30Dec24 and 27Jan25, which means they belong to Jan 2025. They also snap 5 days, which are the only distances above 1 in the whole workbook. Both of them go to zero under 2025 also.

LABELS= {
"Time": "time",
"Food": "food",
"Quantity (g)": "quantity",
"Quantity (g or count)": "quantity",
"Calories": "calories",
"Protein": "protein",
"Fiber": "fibre",
"Carbs": "carbs", 
"Fat": "fat",
}

CANON_NAMES=("time","food","quantity","calories","protein","fibre","carbs","fat") #list of eight column names


assert set(CANON_NAMES) == set(LABELS.values()), f"In CANON_NAMES but not LABELS: {set(CANON_NAMES) - set(LABELS.values())}. In LABELS but not CANON_NAMES: {set(LABELS.values())- set(CANON_NAMES)}."
#


### Normalise the Dates

In [18]:
def date_normalisation(date_raw):    
    if date_raw in OVERRIDES:
        return OVERRIDES[date_raw], True
    pattern=r"(\d{1,2})([A-Za-z]+)(\d{2})$" #using the pattern from earlier
    m = re.match(pattern, date_raw) #m is the match of the pattern with the given date

    day_test=m.group(1) #gives our day date value in accordance with the pattern
    month_test=m.group(2) #month value
    year_test=m.group(3) #year value

    month_normal=month_test[:3] #slices the month to the first 3 characters

    normalised_date=day_test+month_normal+year_test 

    stripped_normalised_date = datetime.strptime(normalised_date, "%d%b%y") #date_processed is the stripped time of the normalised date with the given pattern
    

    return stripped_normalised_date, False




In [19]:
print(date_normalisation("6Jan24"))
print(date_normalisation("14Oct24"))

(datetime.datetime(2025, 1, 6, 0, 0), True)
(datetime.datetime(2024, 10, 14, 0, 0), False)


In [20]:
processed_date=date_normalisation("02June25")
print(processed_date)

week_start=processed_date

(datetime.datetime(2025, 6, 2, 0, 0), False)


Using the date_normalisation function and weekday(), the printed output is the zero-indexed position of the day of the week for the given date. Added the snap to the function above.

In [21]:
date_normalisation("13May25")

(datetime.datetime(2025, 5, 13, 0, 0), False)

### Refine Dates & Snap to Monday

In [22]:
refined_dates=[] #creates a list 
for a_sheet in refined_weeks: #starts the loop
    rd, overriden=date_normalisation(a_sheet) #rd is the result of the function on the var a_sheet
    snapped=rd-(timedelta(days=rd.weekday()))
    distance=(rd-snapped).days
    refined_dates.append((a_sheet, rd, snapped, distance, overriden)) #adds the date and its refined to the list
    assert 0 <= distance <= 6, f"{a_sheet} snapped by {distance} days to {rd}" 
print(len(refined_dates))
print(refined_dates)
    

81
[('14Oct24', datetime.datetime(2024, 10, 14, 0, 0), datetime.datetime(2024, 10, 14, 0, 0), 0, False), ('4Nov24', datetime.datetime(2024, 11, 4, 0, 0), datetime.datetime(2024, 11, 4, 0, 0), 0, False), ('19Nov24', datetime.datetime(2024, 11, 19, 0, 0), datetime.datetime(2024, 11, 18, 0, 0), 1, False), ('25Nov24', datetime.datetime(2024, 11, 25, 0, 0), datetime.datetime(2024, 11, 25, 0, 0), 0, False), ('2Dec24', datetime.datetime(2024, 12, 2, 0, 0), datetime.datetime(2024, 12, 2, 0, 0), 0, False), ('9Dec24', datetime.datetime(2024, 12, 9, 0, 0), datetime.datetime(2024, 12, 9, 0, 0), 0, False), ('16Dec24', datetime.datetime(2024, 12, 16, 0, 0), datetime.datetime(2024, 12, 16, 0, 0), 0, False), ('30Dec24', datetime.datetime(2024, 12, 30, 0, 0), datetime.datetime(2024, 12, 30, 0, 0), 0, False), ('6Jan24', datetime.datetime(2025, 1, 6, 0, 0), datetime.datetime(2025, 1, 6, 0, 0), 0, True), ('20Jan24', datetime.datetime(2025, 1, 20, 0, 0), datetime.datetime(2025, 1, 20, 0, 0), 0, True), ('27

In [23]:
moved_count=0
override_count=0
for name, rd, snapped, distance, overriden in refined_dates:
    if distance != 0:
        print("The sheet ", name, "was moved ", distance, "day(s).")
        moved_count=moved_count+1
    if overriden is True:
        override_count=override_count+1
print(f"{moved_count} of {len(refined_dates)} names required a snap")
print(f"Override Count stands at {override_count}")


The sheet  19Nov24 was moved  1 day(s).
The sheet  13May25 was moved  1 day(s).
The sheet  07APR26 was moved  1 day(s).
3 of 81 names required a snap
Override Count stands at 2


This snapping mechanism is one of the most important elements thus far. The output lists all of the sheet names, original date in datetime Year, Month, Date format, snapped date in datetime format,and how many days the date was moved by the snap. 

('19Nov24', datetime.datetime(2024, 11, 19, 0, 0), datetime.datetime(2024, 11, 18, 0, 0), 1)

Sheet Name: 19Nov24

Date: 2024, 11, 19

Snapped Date: 2024, 11, 18

Days Moved: 1

### Detect Placement and Count+Display if a Sheet's Date is Incorrectly Placed

In [24]:
previous_date = None
previous_name = None
incorrect_placement_count=0

for name, rd, snapped, distance, overriden in refined_dates:
    if previous_date is not None:
        if snapped <= previous_date:
            incorrect_placement_count=incorrect_placement_count+1
            print(f"{name}, dated at {snapped:%Y-%m-%d} is out of place. {name} is preceded by {previous_name}, {previous_date:%Y-%m-%d}")
    previous_date=snapped
    previous_name=name
print(f"Incorrect Placement Count stands at {incorrect_placement_count}")

Incorrect Placement Count stands at 0


This reads for incorrect dates by seeing if it is not later than the date before it. Since everything is normalised to Mondays of the given week, this tells us if theres duplicates (in workbook order) or if a date is incorrect given the placement of the date before it.

Limited by workbook order, this only catches duplicates which are next to one another.

94 sheets in total. 90 of them have log layout. 81 real weeks after removing blanks, samples, and sheets I chose to explicitly ignore. Out of the 81 sheets, three names were a day later than Monday and were snapped accordingly. I had to override 2 of them explicitly to Jan 2025. They were identified by their position in the workbook and by having a 5 day snap, unlike any others. Every other sheet was a Monday. With this correction, no week has a date which precedes the sheet before it. 

### Test for any duplicates

In [25]:
sheet_set=set()

for name, rd, snapped, distance, overriden in refined_dates:
        sheet_set.add(snapped)

total_weeks=len(refined_dates)
distinct_dates=len(sheet_set)

assert distinct_dates == total_weeks, f"{total_weeks} weeks but only {distinct_dates} distinct Mondays, so {total_weeks - distinct_dates} collide"
f"{total_weeks} weeks, {distinct_dates} distinct Mondays, no collisions"

'81 weeks, 81 distinct Mondays, no collisions'

In [26]:
def header_layout_keys(cell_values):   

    header_labels=[]

    for label in cell_values:
        if label is not None:
            header_labels.append(str(label).strip())

    return tuple(header_labels)

In [27]:
rows, values = altered_find_header_rows("06JUL26")

header_layout_keys(values)

('Time',
 'Food',
 'Quantity (g or count)',
 'Calories',
 'Protein',
 'Fiber',
 'Carbs',
 'Fat',
 'Time',
 'Food',
 'Quantity (g or count)',
 'Calories',
 'Protein',
 'Fiber',
 'Carbs',
 'Fat')

In [28]:
header_dict={} #creating empty dictionary
row_check={}
for name in refined_weeks:
    row, values = altered_find_header_rows(name)
    layout= header_layout_keys(values)
    if layout not in row_check:
        row_check[layout] = set()
    row_check[layout].add(row)
    if layout not in header_dict:
        header_dict[layout] = []
    header_dict[layout].append(name)
print(len(header_dict))

for layout, sheets in row_check.items():
    print(len(sheets))

for layout, sheets in header_dict.items():
    print(len(sheets), "weeks with a shared layout. From:", sheets[0], "to", sheets[-1])

3
2
1
1
36 weeks with a shared layout. From: 14Oct24 to 25AUG25
8 weeks with a shared layout. From: 01SEP25 to 03NOV25
37 weeks with a shared layout. From: 10NOV25 to 03AUG26


Initial check using the layouts as keys shows 3 layouts (eras) across the 81 sheets. Grouped on the cleaned header labels. 36 weeks from 14Oct24 to 25AUG25, 8 from 01SEP25 to 03NOV25, 37 from 10NOV to 03AUG26. No overlap.

However, secondary heck shows that for 14Oct24 to 25AUG25 sheets, the header sits on row 2 for some and 3 for others. This means there are really 4 layouts, as some of the era 1 sheets sit a row lower.

So, grouping on labels alone reports 3 layouts. Row-number checking reports the first group spanning 2 header rows. The parser must key on both the label and the row number.

In [29]:
header_dict={} #creating empty dictionary

for name in refined_weeks:

    row, values = altered_find_header_rows(name)
    layout = header_layout_keys(values)
    key = (layout, row)

    if  key not in header_dict:
        header_dict[key] = []
    header_dict[key].append(name)
print(len(header_dict))

for layout, sheets in header_dict.items():
    print(len(sheets), "weeks with a shared layout. From:", sheets[0], "to", sheets[-1])

4
23 weeks with a shared layout. From: 14Oct24 to 13May25
13 weeks with a shared layout. From: 17Feb25 to 25AUG25
8 weeks with a shared layout. From: 01SEP25 to 03NOV25
37 weeks with a shared layout. From: 10NOV25 to 03AUG26


Keying on layout and row as key splits 4 groups.

In [30]:
for name in refined_weeks:
    rows, values = altered_find_header_rows(name)
    print(name, rows)

14Oct24 2
4Nov24 2
19Nov24 2
25Nov24 2
2Dec24 2
9Dec24 2
16Dec24 2
30Dec24 2
6Jan24 2
20Jan24 2
27Jan25 2
03Feb25 2
10Feb25 2
17Feb25 3
24Feb25 2
03Mar25 2
10Mar25 2
17Mar25 2
24Mar25 2
31Mar25 2
07Apr25 2
14Apr25 2
21Apr25 3
28Apr25 2
05May25 3
13May25 2
19May25 3
26May25 3
02June25 3
09June25 3
30JUN25 3
07JUL25 3
14JUL25 3
11AUG25 3
18AUG25 3
25AUG25 3
01SEP25 3
08SEP25 3
15SEP25 3
06OCT25 3
13OCT25 3
20OCT25 3
27OCT25 3
03NOV25 3
10NOV25 3
17NOV25 3
24NOV25 3
01DEC25 3
08DEC25 3
29DEC25 3
05JAN26 3
12JAN26 3
19JAN26 3
26Jan26 3
02Feb26 3
09Feb26 3
16Feb26 3
23Feb26 3
02Mar26 3
09Mar26 3
16Mar26 3
23Mar26 3
30Mar26 3
07APR26 3
13APR26 3
20APR26 3
27APR26 3
04MAY26 3
11MAY26 3
18MAY26 3
25MAY26 3
01JUN26 3
08JUN26 3
15JUN26 3
22JUN26 3
29JUN26 3
06JUL26 3
13JUL26 3
20JUL26 3
27JUL26 3
03AUG26 3


This shows that within our different eras, the reason for overlap in two groups, where the header is on row 3 or row 2, is due to interleaving of these row positions. The 2,3,2,3 is exactly this. 

In [31]:
ws=wb["06JUL26"]

sheet_rows=list(ws.iter_rows(values_only=True))

print(len(sheet_rows))
print(sheet_rows[2])

243
(None, None, 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', None, 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)


This is to confirm zero indexing. Index 2 is Excel row 3.
243 rows. The last Total is at row 108. Most likely blank rows.

In [32]:
field_map={} #empty dictionary
header = sheet_rows[2] #header is set as index 2 (excel row 3) of the sheet
start_column = 2 #hardcoded start 3 columns in (zero indexing)

for column_number, cell_value in enumerate(header):
    if column_number < start_column:
        continue #... then skip this iteration of the loop and start the next one.
    if cell_value is None:
        break
    field_map[cell_value] = column_number #cell_value, such as Time, is what we know exists. We need to know what column it is in, so cell_value must be the key we use to look up the column number.

print(field_map)

{'Time': 2, 'Food': 3, 'Quantity (g or count)': 4, 'Calories': 5, 'Protein': 6, 'Fiber': 7, 'Carbs': 8, 'Fat': 9}


In [33]:
parsed_rows=[]

for row in sheet_rows[3:25]:
    if row[field_map['Food']] is None:
        continue
    row_dict = {'food_value':row[field_map['Food']], 'calorie_value':row[field_map['Calories']],'quant_value':row[field_map['Quantity (g or count)']],'protein_value':row[field_map['Protein']],'fibre_value':row[field_map['Fiber']],'carbs_value':row[field_map['Carbs']],'fat_value':row[field_map['Fat']],'time_value':row[field_map['Time']]}
    parsed_rows.append(row_dict)
print(parsed_rows)

[{'food_value': 'Diced Steak (100g)', 'calorie_value': 460, 'quant_value': 4, 'protein_value': 85.6, 'fibre_value': 0, 'carbs_value': 0, 'fat_value': 12, 'time_value': None}, {'food_value': 'Deli Wrap, breaded fillet', 'calorie_value': 710, 'quant_value': 1, 'protein_value': 35, 'fibre_value': 5, 'carbs_value': 70, 'fat_value': 30, 'time_value': None}, {'food_value': 'Avonmore Protein Gold Milk', 'calorie_value': 275, 'quant_value': 1, 'protein_value': 34, 'fibre_value': 0, 'carbs_value': 24.5, 'fat_value': 5, 'time_value': None}]


### Extraction


In [54]:
bands = map_bands(sheet_rows)
time_rows = find_label_rows(sheet_rows, "Time")
parsed_rows=[]
block_count=0

rd, overriden = date_normalisation("06JUL26")
week_monday = rd - (timedelta(days=rd.weekday()))
print(week_monday)

for header_row, total_row in bands.items():
    
    for start_column in time_rows[header_row]:
        block_index=block_count
        block_date=week_monday+timedelta(days=block_index)
        field_map = map_fields(sheet_rows, header_row, start_column, time_rows[header_row])

        for row in sheet_rows[header_row: total_row - 1]:
            if row[field_map['food']] is None:
                continue
            row_dict={}
            for name in CANON_NAMES:
                if name in field_map:
                    row_dict[name] = row[field_map[name]]
                else: 
                    row_dict[name]= None
            row_dict['date']= block_date
            row_dict['weekday'] = block_date.strftime("%A")
            row_dict['source_sheet'] = "06JUL26"
            parsed_rows.append(row_dict)
        block_count = block_count+1
        print(len(parsed_rows))
        print(block_count)
        print(block_date)
assert block_date == week_monday + timedelta(days=6), f'Block Date = {block_date}, should have been {week_monday + timedelta(days=6)}'
print(parsed_rows[0])
print(parsed_rows[20])


2026-07-06 00:00:00
3
1
2026-07-06 00:00:00
12
2
2026-07-07 00:00:00
24
3
2026-07-08 00:00:00
34
4
2026-07-09 00:00:00
45
5
2026-07-10 00:00:00
45
6
2026-07-11 00:00:00
45
7
2026-07-12 00:00:00
{'time': None, 'food': 'Diced Steak (100g)', 'quantity': 4, 'calories': 460, 'protein': 85.6, 'fibre': 0, 'carbs': 0, 'fat': 12, 'date': datetime.datetime(2026, 7, 6, 0, 0), 'weekday': 'Monday', 'source_sheet': '06JUL26'}
{'time': datetime.time(18, 30), 'food': 'Skinless Pollock (100g)', 'quantity': 1.4, 'calories': 99.39999999999999, 'protein': 22.4, 'fibre': 0, 'carbs': 0, 'fat': 1.1199999999999999, 'date': datetime.datetime(2026, 7, 8, 0, 0), 'weekday': 'Wednesday', 'source_sheet': '06JUL26'}


### Find Label Rows

In [38]:
def find_label_rows(sheet_rows, target): 
    found_rows={}
    for row_number, cell_values in enumerate(sheet_rows, start=1): 
        given_columns = [] 
        for column_number, value in enumerate(cell_values): 
            if str(value).strip() == target: 
                given_columns.append(column_number) 
        if given_columns: 
            found_rows[row_number]=given_columns
    return found_rows

Version of the find_header_rows functions from earlier. These served their purpose for discovery, but this version will be in the final version as it returns the values instead of printing, and walks the whole sheet telling us where the given second parameter is throughout.

These two should be considered superseded. Can take the data from this, the parameter positions, and feed it into a loop. We will do this on 3 layers for sheet, band, and block later. This way it runs through the whole sheet, finding the positions of each band. Then it runs through each band, finding the position of each block. Then it runs through each block, or in other words it walks the row range.

This will get the food dictionaries, telling us at each given spot, what is there. For each row in the range, if there is a food name, build a dictionary recording what is there. Burger, 300 calories, 50 protein.

Function is called twice per sheet, once with Time, and once with Total. Gets every band's start and end row.

Returned keys are Excel row numbers, because of start=1 in the enumeration. Sheets_row is zero-indexed, so excel row three is sheets_row[2].

In [ ]:
print(find_label_rows(sheet_rows, "Time"))
print(find_label_rows(sheet_rows, "Total"))

{3: [2, 11], 31: [2, 11], 58: [2, 11], 85: [2]}
{26: [2, 11], 54: [2, 11], 81: [2, 11], 108: [2]}


In [ ]:
ws=wb["2Dec24"]

Second_Dec24_sheet_rows=list(ws.iter_rows(values_only=True))

print(find_label_rows(Second_Dec24_sheet_rows, "Time"))
print(find_label_rows(Second_Dec24_sheet_rows, "Total"))

{2: [1, 6], 21: [1, 6], 40: [1, 6], 59: [1]}
{18: [1, 6], 37: [1, 6], 56: [1, 6], 75: [1]}


Testing on 2Dec24

In [ ]:
for (labels, row), sheets in header_dict.items():
    print(row, labels)

2 ('Time', 'Food', 'Calories', 'Protein', 'Time', 'Food', 'Calories', 'Protein')
3 ('Time', 'Food', 'Calories', 'Protein', 'Time', 'Food', 'Calories', 'Protein')
3 ('Time', 'Food', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', 'Time', 'Food', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat')
3 ('Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat', 'Time', 'Food', 'Quantity (g or count)', 'Calories', 'Protein', 'Fiber', 'Carbs', 'Fat')


Layout 1+2 have Time, Food, Calories, Protein. 4x fields

Layout 3 adds Fiber, Carbs, Fat. 7x fields

Layout 4 adds Quantity. 8x field

No spelling variations, translation table only required for quantity and fiber. 36 of 81 weeks only contain calories and protein. Full macro coverage is 45 weeks, with quantity coverage at 37.

For each header row, look through the Total rows, find the first one that is below it and has the same columns. Record the pair, stop looking.


### Map Bands


In [36]:
def map_bands(sheet_rows):

    time_rows = find_label_rows(sheet_rows, "Time")
    total_rows = find_label_rows(sheet_rows, "Total")

    result_dict={}

    for (header_row_no, header_column_list) in time_rows.items():
        for (total_row_no, total_column_list) in  total_rows.items():
            if total_row_no>header_row_no and header_column_list==total_column_list:
                result_dict[header_row_no]=total_row_no
                break
    assert len(result_dict)==len(time_rows), f"{len(time_rows)} headers expected. {len(result_dict)} found. {len(time_rows)-len(result_dict)} unpaired." 
    return(result_dict)


In [ ]:
ws=wb["13OCT25"]

Thirteen_Oct_sheet_rows=list(ws.iter_rows(values_only=True))

In [ ]:
print(map_bands(sheet_rows))
print(map_bands(Thirteen_Oct_sheet_rows))

{3: 26, 31: 54, 58: 81, 85: 108}
{3: 26, 31: 54, 58: 81, 87: 110}


This finds the four day-block regions on any sheet it is given. 

Find_label_rows walks the sheet for a given label, returning every row it appears on and the columns it is on. Using "Time" gives the 4 header rows, and Total gives the four totals.

The function joins together the two, finding the first Total below each header row where the columns match, and pairs them up together. {3:26} means band 1 starts on the third row in excel and ends on the 26th row. Pairing by order would work on a sheet without any impurities, but 13OCT25 has a Total at row 12 column 21, which does not belong to anything, sitting above all the real ones. So if pairing was done by order, band 1 would have end row 12 instead of 26. The assertion essentially creates an error message for the function. 

To conclude, previously the bounds of the bands(the row range we are extracting) were typed by hand, but now I can get them from this function. Band 1 has food rows at row 4 to 25, header+1 up to the total. Field map still has sheet_rows[2] and start_column=2, this is next on the list.

In [40]:
def map_fields(sheet_rows, header_row_no, start_column, column_list):
    field_map={} 
    header = sheet_rows[header_row_no-1] #sheet_rows is zero indexed, so minus 1 for the excel row.

    position = column_list.index(start_column)

    if position+1 < len(column_list):
        bound = column_list[position+1]-1 #-1 so it stops before the next block. One blank between blocks assumed
    else: 
        bound = len(header)

    for column_number, cell_value in enumerate(header):
        if column_number < start_column:
            continue
        if column_number>=bound:
            break
        if cell_value is None:
            break

        if cell_value not in LABELS:
            raise ValueError(f"Label '{cell_value}' at header row {header_row_no}, and start column {start_column} not recognised.")
        else: 
            field_map[LABELS[cell_value]] = column_number



    return(field_map)

In [ ]:
print(map_fields(sheet_rows, 3, 2,))
print(map_fields(sheet_rows, 3, 11))
print(map_fields(sheet_rows, 31, 2))
print(map_fields(sheet_rows, 85, 2))

TypeError: map_fields() missing 1 required positional argument: 'column_list'

This exposed the spelling difference in Quantity, which I previously thought solved.

In [ ]:
labels = set()

for week in refined_weeks:    
    ws=wb[week]
    current_sheet_rows=list(ws.iter_rows(values_only=True))

    bands = map_bands(current_sheet_rows)
    time_rows=find_label_rows(current_sheet_rows, "Time")

    for header_row_no, column_list in time_rows.items():
        for start_column in column_list:
            field_map_here=map_fields(current_sheet_rows, header_row_no, start_column, column_list)
            labels.update(field_map_here.keys())
            for key in field_map_here.keys(): #Tests the field maps keys, walking them one at  a time and assigning to key.
                if not isinstance(key, str): #tests if the keys are strings, and if not:
                    print(week, header_row_no, start_column, key) #tells the details
print(labels)

NameError: name 'refined_weeks' is not defined

Walks across all blocks start column and to the right, stopping at each blank. Does this for every sheet, and found the two Quantity spellings and 3184

Altered to find the header row no and start column of 3184. Opened excel file to find it. Must rework the bounds rule to not just be at a blank.

In [ ]:
ws=wb["10Feb25"]
feb_sheet_rows = list(ws.iter_rows(values_only=True))
map_fields(feb_sheet_rows, 21, 1, [1,6])

5


{'Time': 1, 'Food': 2, 'Calories': 3, 'Protein': 4}

In [73]:
parsed_rows = []
short_weeks = []
report_rows = []


for week in refined_weeks:
    ws = wb [week]
    local_sheet_rows=list(ws.iter_rows(values_only=True))

    bands = map_bands(local_sheet_rows)
    time_rows = find_label_rows(local_sheet_rows, "Time")
    

    rd, overriden = date_normalisation(week)
    week_monday = rd - (timedelta(days=rd.weekday()))

    block_count=0

    for header_row, total_row in bands.items():
        for start_column in time_rows[header_row]:
            calories_count=0
            protein_count=0
            missing_calories=0
            missing_protein=0

            block_index=block_count
            block_date=week_monday+timedelta(days=block_index)
            field_map = map_fields(local_sheet_rows, header_row, start_column, time_rows[header_row])
            

            for row in local_sheet_rows[header_row: total_row - 1]:
                if row[field_map['food']] is None:
                    continue
                row_dict={}
                for name in CANON_NAMES:
                    if name in field_map:
                        row_dict[name] = row[field_map[name]]
                    else: 
                        row_dict[name]= None
                row_dict['date']= block_date
                row_dict['weekday'] = block_date.strftime("%A")
                row_dict['source_sheet'] = week
                parsed_rows.append(row_dict)


                if isinstance(row_dict['calories'], (int, float)):
                    calories_count=calories_count + row_dict['calories']
                else:
                    print("Calories", week, block_date, row_dict['food'])
                    missing_calories=missing_calories+1

                if isinstance(row_dict['protein'], (int, float)):
                    protein_count=protein_count + row_dict['protein']
                else:
                    print("Protein", week, block_date, row_dict['food'])
                    missing_protein=missing_protein+1
                
    

            sheet_total_calories=local_sheet_rows[total_row-1][field_map['calories']]
            sheet_total_protein=local_sheet_rows[total_row-1][field_map['protein']]

            if isinstance(sheet_total_calories, (int, float)):
                calories_diff=calories_count-sheet_total_calories
            else:
                calories_diff=None
                print("Calorie Issue found: ", week, block_date, calories_count)

            if isinstance(sheet_total_protein, (int, float)):
                protein_diff=protein_count-sheet_total_protein
            else:
                protein_diff=None
                print("Protein Issue found: ", week, block_date, protein_count)
            
            ## report_rows.append()
            report_rows.append({
                                'source_sheet': week,
                                'date': block_date,
                                'metric': "calories",
                                'parsed': calories_count,
                                'expected': sheet_total_calories,
                                'diff': calories_diff
                                })

            report_rows.append({
                                'source_sheet': week,
                                'date': block_date,
                                'metric': "protein",
                                'parsed': protein_count,
                                'expected': sheet_total_protein,
                                'diff': protein_diff
                                })
           
            block_count = block_count+1
        
    assert block_count <= 7, f'{week} has {block_count} blocks.'
    if block_count<7:
        short_weeks.append((week, block_count))

print(len(report_rows))
    

        

Protein 14Oct24 2024-10-16 00:00:00 snack
Protein 14Oct24 2024-10-17 00:00:00 Lucozade energy
Protein 4Nov24 2024-11-09 00:00:00 butter
Protein 4Nov24 2024-11-09 00:00:00 coffee
Protein 4Nov24 2024-11-10 00:00:00 g. mayo
Protein 4Nov24 2024-11-10 00:00:00 2x oreos
Protein 19Nov24 2024-11-19 00:00:00 sauce
Protein 19Nov24 2024-11-21 00:00:00 bread roll
Protein 19Nov24 2024-11-21 00:00:00 croissant
Protein 19Nov24 2024-11-21 00:00:00 pt milk
Protein 19Nov24 2024-11-21 00:00:00 4x biscuits
Protein 19Nov24 2024-11-21 00:00:00 excess
Protein 19Nov24 2024-11-22 00:00:00 apple
Protein 19Nov24 2024-11-23 00:00:00 frozen veg
Protein 19Nov24 2024-11-23 00:00:00 sauces
Protein 19Nov24 2024-11-23 00:00:00 snickers
Protein 19Nov24 2024-11-24 00:00:00 3x eggs
Protein 19Nov24 2024-11-24 00:00:00 2x slice bread
Protein 19Nov24 2024-11-24 00:00:00 excess
Protein 25Nov24 2024-11-25 00:00:00 2x small apples
Calories 25Nov24 2024-11-25 00:00:00 rice
Protein 25Nov24 2024-11-25 00:00:00 rice
Calories 25Nov2

parsed_rows and short_weeks exist at the top, outside the loops. They will build each time across the entire workbook.

* Sheets: For each one of the 81 week names, the sheet's rows are loaded. Bands are found, Time columns are found. The week's monday is found from the sheet name, and the block counter is then reset to zero. The monday comes from date_normalisation, which is what sorts out month spellings + the Jan overrides. It also snaps 3 of the sheets which were a day later than actuality.

* Bands: Header row and Total row are paired by the column. This is to ensure odd cells such as the Total from 13OCT25 are excluded.

* Blocks: The position of a block for a given Time column in a band becomes the date, which is Monday + the index of the block. Field map built from the band's header row, boundary set by the next block's start, and the keys are set by CANON_NAMES through LABELS.

* Rows: For every given row between Header and Total, the cell is skipped if it is empty. If it is not empty, the dictionary is built using the cell's value by looping the CANON_NAMES and taking the value wherever the field exists on the layout, with None taken if it doesn't exist. The date is then attached, with the weekday derived from the date and the source sheet. This is appended.

* After each sheet: Assertion for sheets which have less than 7 blocks. These are recorded.

In [60]:
ws=wb["06JUL26"]
rows_list=list(ws.iter_rows(values_only=True))

bands = map_bands(rows_list)
field_map=map_fields(rows_list,3, 2, [2,11])

print(rows_list[bands[3]-1][field_map['calories']])


1445


In [78]:
import pandas as pd

pd.DataFrame(report_rows).to_csv("../reports/validation.csv", index=False)
pd.DataFrame(parsed_rows).to_csv("../data/processed/parsed_rows.csv", index=False)

In [79]:
v = pd.DataFrame(report_rows)
v[(v['parsed'] == 0) & (v['expected'] != 0)].head(20)

,source_sheet,date,metric,parsed,expected,diff
78,9Dec24,2024-12-13,calories,0.0,3500,-3500.0
82,9Dec24,2024-12-15,calories,0.0,4500,-4500.0
96,16Dec24,2024-12-22,calories,0.0,2700,-2700.0
98,30Dec24,2024-12-30,calories,0.0,3800,-3800.0
110,30Dec24,2025-01-05,calories,0.0,2800,-2800.0
222,03Mar25,2025-03-09,calories,0.0,5000,-5000.0
234,10Mar25,2025-03-15,calories,0.0,200,-200.0
248,17Mar25,2025-03-22,calories,0.0,200,-200.0
258,24Mar25,2025-03-27,calories,0.0,1070,-1070.0
259,24Mar25,2025-03-27,protein,0.0,70,-70.0
